# 01 — Detección de phishing con PhiUSIIL
### Tema 6 — Sistemas expertos en SOC | IA Aplicada a la Ciberseguridad

Este notebook entrena y evalúa el clasificador de phishing que alimenta la rama izquierda de la
arquitectura del proyecto (ver README del repositorio).

**Dataset:** PhiUSIIL Phishing URL Dataset — UCI Machine Learning Repository (2024)
https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset

**Referencia académica:**
Prasad, A. & Chandra, S. (2024). PhiUSIIL: A diverse security profile empowered phishing URL detection
framework based on similarity index and incremental learning. Computers & Security.
https://doi.org/10.1016/j.cose.2023.103545

**MITRE ATT&CK relacionado:** T1566 (Phishing), T1566.001 (Spearphishing Attachment), T1204.002 (User
Execution: Malicious File) — https://attack.mitre.org/techniques/T1566/


## 1. Instalación de dependencias

In [ ]:
# Descomenta en la primera ejecucion del entorno
# !pip install ucimlrepo xgboost shap scikit-learn matplotlib seaborn joblib --quiet


## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_fscore_support, roc_auc_score, RocCurveDisplay
)
import xgboost as xgb
import shap
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 3. Carga del dataset (entrada del pipeline)

`fetch_ucirepo` descarga el dataset directamente desde el repositorio UCI — no necesitas subir archivos
manualmente a Kaggle/Colab.

In [ ]:
dataset = fetch_ucirepo(id=967)  # PhiUSIIL Phishing URL Dataset

X_raw = dataset.data.features
y_raw = dataset.data.targets

print(dataset.metadata.name)
print(dataset.metadata.abstract if hasattr(dataset.metadata, "abstract") else "")
print("Shape features:", X_raw.shape)
print("Shape target:", y_raw.shape)

df = X_raw.copy()
df["label"] = y_raw.values.ravel()
df.head()


## 4. Limpieza básica

In [ ]:
# La columna FILENAME es un identificador, no una feature (documentado en la ficha del dataset)
if "FILENAME" in df.columns:
    df = df.drop(columns=["FILENAME"])

df = df.drop_duplicates()
df = df.dropna()

print(df.shape)
print(df["label"].value_counts())


## 5. Feature importance (para justificar la selección de features en el informe)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

y_all = df["label"]
X_all = df.drop(columns=["label"]).select_dtypes(include=[np.number])

X_train_tmp, X_test_tmp, y_train_tmp, y_test_tmp = train_test_split(
    X_all, y_all, test_size=0.3, random_state=RANDOM_STATE, stratify=y_all
)

rf_probe = RandomForestClassifier(n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1)
rf_probe.fit(X_train_tmp, y_train_tmp)

importances = pd.Series(rf_probe.feature_importances_, index=X_all.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 7))
importances.head(20).plot(kind="barh")
plt.gca().invert_yaxis()
plt.xlabel("Importancia (Gini)")
plt.title("Top 20 features - PhiUSIIL - Random Forest")
plt.tight_layout()
plt.savefig("phishing_feature_importance.png", dpi=150)
plt.show()

importances.head(20)


## 6. Selección final de features

Ajusta esta lista con los nombres reales que veas en `importances.head(20)` de tu ejecución — PhiUSIIL
incluye features léxicas de la URL (longitud, uso de guiones, IP en dominio), de contenido HTML
(iframes, favicons externos, formularios ocultos) y de reputación (edad del dominio, presencia en
motores de búsqueda).

In [ ]:
candidate_features = [
    "URLLength", "DomainLength", "IsDomainIP", "TLDLength",
    "NoOfSubDomain", "HasObfuscation", "NoOfLettersInURL", "NoOfDigitsInURL",
    "NoOfEqualsInURL", "NoOfQMarkInURL", "NoOfAmpersandInURL",
    "IsHTTPS", "LineOfCode", "LargestLineLength",
    "HasFavicon", "HasExternalFormSubmit", "HasHiddenFields",
    "NoOfImage", "NoOfCSS", "NoOfJS", "NoOfSelfRef", "NoOfEmptyRef", "NoOfExternalRef",
]

selected_features = [f for f in candidate_features if f in df.columns]
print(f"Features finales usadas ({len(selected_features)}):")
print(selected_features)

X = df[selected_features]
y = df["label"]


## 7. Split y escalado

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(dict(zip(le.classes_, range(len(le.classes_)))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.25, random_state=RANDOM_STATE, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 8. Entrenamiento (XGBoost)

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(X_train_scaled, y_train)


## 9. Evaluación (precisión, recall, F1, AUC)

In [ ]:
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred, target_names=[str(c) for c in le.classes_]))

precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted")
auc = roc_auc_score(y_test, y_proba)

metrics_table = pd.DataFrame({
    "Metrica": ["Precision", "Recall", "F1-score", "AUC"],
    "Valor": [round(precision, 4), round(recall, 4), round(f1, 4), round(auc, 4)],
})
metrics_table


In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[str(c) for c in le.classes_])

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Oranges", colorbar=True)
plt.title("Matriz de confusion - Deteccion de phishing (PhiUSIIL)")
plt.tight_layout()
plt.savefig("phishing_confusion_matrix.png", dpi=150)
plt.show()


In [ ]:
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("Curva ROC - Deteccion de phishing")
plt.tight_layout()
plt.savefig("phishing_roc_curve.png", dpi=150)
plt.show()


## 10. Interpretabilidad (SHAP)

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_scaled[:500])

shap.summary_plot(
    shap_values, X_test_scaled[:500],
    feature_names=selected_features,
    show=False,
)
plt.tight_layout()
plt.savefig("phishing_shap_summary.png", dpi=150)
plt.show()


## 11. Guardado del modelo (usado por el notebook 03 de fusión)

In [ ]:
joblib.dump(model, "phishing_model.joblib")
joblib.dump(scaler, "phishing_scaler.joblib")
joblib.dump(le, "phishing_label_encoder.joblib")
joblib.dump(selected_features, "phishing_selected_features.joblib")

print("Modelo de phishing guardado. Cópialo junto al notebook 03 antes de ejecutarlo.")
